# Week 1 — 報酬、風險與線性代數

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 正確計算 simple return、log return 與累積報酬。
- 計算年化平均、年化波動度、共變異數與相關矩陣。
- 用 quadratic form $w^\top\Sigma w$ 計算投資組合變異數。
- 理解特徵值、特徵向量與 positive semidefinite (PSD)。

## 預估學習時間

約 8–10 小時。

## 先備概念

- 向量與矩陣運算
- 平均與變異數的定義

## 外部學習資源

- [MIT OpenCourseWare 18.06SC Linear Algebra](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/)
- [NTU OpenCourseWare 基礎財金素養](https://ocw.aca.ntu.edu.tw/courses/110S204)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

In [ ]:
# 教學樣式設定（CJK 字型、負號正常顯示、固定隨機種子）
import matplotlib as _mpl
_mpl.rcParams['font.sans-serif'] = [
    'PingFang TC', 'Heiti TC', 'Microsoft JhengHei',
    'Noto Sans CJK TC', 'Noto Sans TC',
    'WenQuanYi Zen Hei', 'Source Han Sans TC',
    'Arial Unicode MS', 'DejaVu Sans',
]
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## 概念說明

### 報酬

給定價格 $P_t$，**simple return** 與 **log return** 定義為：

$$ r_t = \frac{P_t - P_{t-1}}{P_{t-1}}, \qquad \ell_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right). $$

log return 是**時間可加**的：多期 log return 等於各期之和。

### 投資組合變異數

設權重向量 $w$、共變異數矩陣 $\Sigma$，投資組合變異數為 quadratic form：

$$ \operatorname{Var}(r_p) = w^\top \Sigma\, w. $$

因為變異數不可能為負，對任意 $w$ 都有 $w^\top\Sigma w \ge 0$，這正是「$\Sigma$ 必須是 **positive semidefinite (PSD)**」的意義——等價於它所有特徵值 $\ge 0$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.data import SyntheticConfig, generate_correlated_prices
from quant_math_roadmap.finance.returns import simple_returns, log_returns
from quant_math_roadmap.finance.metrics import (
    annualized_mean, annualized_volatility,
    covariance_matrix, correlation_matrix,
)
from quant_math_roadmap.finance.portfolio import equal_weights, portfolio_variance
from quant_math_roadmap.math.linear_algebra import (
    eigendecomposition, is_positive_semidefinite,
)

config = SyntheticConfig(n_assets=4, n_periods=756, seed=11,
                         average_correlation=0.4)
prices = generate_correlated_prices(config)
prices.tail()

### 計算報酬並比較 simple vs log

In [ ]:
simple = simple_returns(prices)
log = log_returns(prices)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(simple.index, simple.iloc[:, 0], label='simple return')
ax.plot(log.index, log.iloc[:, 0], label='log return')
ax.set_title(f'{prices.columns[0]}：simple vs log return')
ax.set_xlabel('日期')
ax.set_ylabel('每日報酬')
ax.legend()
plt.show()

兩者在日報酬尺度上幾乎重疊；差異在報酬較大時才明顯。log return 的好處是可加性，對後面的迴歸與時間序列模型很方便。

### 年化平均與波動度（注意：年化是一個明示的假設）

In [ ]:
ann_mean = annualized_mean(simple, frequency='daily')
ann_vol = annualized_volatility(simple, frequency='daily')
summary = ann_mean.to_frame('年化平均').join(ann_vol.to_frame('年化波動度'))
summary

我們**明示**把 `frequency='daily'`（每年 252 個交易日）傳進去。若資料其實是週資料卻用 252 年化，波動度會被高估約 $\sqrt{252/52}\approx 2.2$ 倍。

### 共變異數、相關矩陣與 quadratic form

In [ ]:
cov = covariance_matrix(simple)
corr = correlation_matrix(simple)
print('共變異數矩陣:')
print(cov.round(6))
print('\n相關矩陣:')
print(corr.round(3))

In [ ]:
weights = equal_weights(prices.shape[1])
# 手算 quadratic form w^T Sigma w
manual = float(weights @ cov.to_numpy() @ weights)
# 用可重用函式
via_function = portfolio_variance(weights, cov.to_numpy())
print(f'手算投組變異數     = {manual:.8f}')
print(f'函式計算投組變異數 = {via_function:.8f}')
assert np.isclose(manual, via_function)

### 特徵值與 PSD 驗證

In [ ]:
eigenvalues, eigenvectors = eigendecomposition(cov.to_numpy())
print('共變異數矩陣特徵值:', np.round(eigenvalues, 8))
print('是否 PSD（所有特徵值 >= 0）:', is_positive_semidefinite(cov.to_numpy()))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(eigenvalues) + 1), eigenvalues)
ax.set_title('共變異數矩陣的特徵值')
ax.set_xlabel('特徵值索引')
ax.set_ylabel('特徵值')
plt.show()

所有特徵值都 $\ge 0$，因此共變異數矩陣是 PSD。最大的特徵值對應的特徵向量，常被解讀為資產間最主要的共同變動方向（與 PCA 相關）。

### 若估計矩陣不是 PSD 怎麼辦？

有時雜訊或浮點誤差會讓共變異數估計出現極小的負特徵值。`nearest_psd()` 會把負特徵值截斷到 0（或某個下界 epsilon），投影到最近的 PSD 矩陣。這是教學級的修補，不是 shrinkage 的替代方案。

In [ ]:
from quant_math_roadmap.math.linear_algebra import nearest_psd

# 故意把一個小的負特徵值灌進共變異數矩陣
noisy = cov.to_numpy().copy()
noisy[0, 0] -= 2 * eigenvalues.max()  # 強迫產生負特徵值
print('修補前最小特徵值:', round(float(np.linalg.eigvalsh(noisy).min()), 6))
repaired = nearest_psd(noisy, epsilon=1e-8)
print('修補後最小特徵值:', round(float(np.linalg.eigvalsh(repaired).min()), 8))

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用文字說明為什麼 log return 可加、而 simple return 不可加。
2. 解釋若某個共變異數矩陣有一個負的特徵值，會代表什麼不合理的情況。
3. 為什麼相關矩陣的對角線一定是 1？

### 應用練習

In [ ]:
# 應用練習 1：不要用 pct_change，從頭用 numpy 實作 simple return，
# 並與 simple_returns() 的結果比對。
p = prices.iloc[:, 0].to_numpy()
my_simple = None  # TODO: (p[1:] - p[:-1]) / p[:-1]
if my_simple is not None:
    print('最大誤差:', np.max(np.abs(my_simple - simple.iloc[:, 0].to_numpy())))

In [ ]:
# 應用練習 2：比較 equal-weight 投組變異數 與「只買波動度最低資產」的變異數。
# 哪一個比較低？為什麼分散投資通常有幫助？
eq_var = portfolio_variance(equal_weights(prices.shape[1]), cov.to_numpy())
lowest_vol_idx = None  # TODO: int(np.argmin(np.diag(cov.to_numpy())))
print('equal-weight 變異數:', eq_var)
print('（完成 TODO 後印出單一資產變異數）')

### 反思問題

1. 共變異數矩陣是用**歷史**資料估計的。若把它直接用來預測**未來**的投組風險，可能出什麼問題？這對回測有什麼啟示？

## 小測驗（自我檢核）
回答下面的選擇題，然後執行下一格自動對答案。答案以雜湊儲存，不會直接洩漏。

**Q1. 投資組合變異數的矩陣公式是？**
- A. wᵀΣw
- B. wᵀμ
- C. Σw
- D. wwᵀ

**Q2. 共變異數矩陣必須是 PSD，原因是？**
- A. 它是對稱矩陣
- B. 任何投組的變異數 wᵀΣw 都不可能為負
- C. 特徵值必須為整數
- D. 為了數值穩定

**Q3. log return 相對 simple return 的關鍵性質是？**
- A. 永遠較大
- B. 跨期可加
- C. 不受價格影響
- D. 永遠為正

**Q4. 把日波動度年化要乘上？**
- A. 252
- B. √252
- C. 12
- D. √12

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: 填入 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '6bd2b1bae6008812', 2: 'bcd9ed8e382c41c5', 3: 'e703c3c3c9cc6729', 4: '21bfb83f4954b4fd'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: 未作答')
        continue
    _h = _hashlib.sha256(f'qmr-w1-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ 正確' if _ok else '✘ 不正確'))
print(f'得分: {_n_correct} / {len(my_answers)}')

## 常見錯誤

- **忘記報酬序列比價格序列少一個觀測值（第一天沒有報酬）。**
- **年化時沒有明示資料頻率，預設所有資料都是日資料。**
- **對非對稱或非方陣呼叫特徵分解。**
- **用樣本共變異數矩陣時忽略它只是估計值、本身有誤差。**

## 完成本週後，你應該能做到什麼

- [ ] 能正確計算 simple/log return 並解釋差異。
- [ ] 能計算年化平均與波動度，並說明年化假設。
- [ ] 能用 $w^\top\Sigma w$ 計算投組變異數。
- [ ] 能檢查一個矩陣是否為 PSD 並解釋其意義。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。